> ## ⚠️  Superseded — single-run draft, retained as a record
>
> **This notebook will not run.** It opens `processed_data/all_data.pkl`, the earlier
> 53-feature preprocessing artefact that was replaced by `canonical_data.pkl` when the
> feature set was corrected. Nothing in this folder regenerates it, so the first cell raises
> `FileNotFoundError`.
>
> **No result reported in the thesis comes from this notebook.** Every reported figure is
> produced by the canonical path `00`/`01` → `08`–`15`, which trains over thirty fixed seeds
> (`SEEDS = [42 + 7*i for i in range(30)]`). Model definitions — `TemporalAttention`,
> `build_lstm_transformer`, `build_standalone_lstm` — live in `canonical_pipeline.py`.
>
> It is kept because its saved outputs are the record of the single-run baseline stage the
> project passed through. See README §5.
>
> **Why this one matters.** The markdown in the training cell documents the feature-leakage
> fix: an earlier Random Forest could see case-history columns and scored R² = 0.93; after the
> fix it scores R² = −0.35, which is the figure reported in the thesis.

# 03 — Train Random Forest
ML baseline model (Section 2.2.2)

In [1]:
import pickle, numpy as np, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

with open('processed_data/all_data.pkl', 'rb') as f:
    data = pickle.load(f)
X_train = data['X_train_scaled']
X_test = data['X_test_scaled']
y_train = data['y_train']
y_test = data['y_test']
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (2888, 53), Test: (723, 53)


### ⚠️ Audit fix — Random Forest temporal leakage (CONFIRMED via real re-run)

The original training cell fit the model on same-day `X_train`/`y_train` (no horizon shift), so the model could see same-day inputs when predicting same-day case counts. This produced an unrealistic RMSE=13.94 / R²=0.9375 that Notebook 07 cited as the **"best overall" model**. A first fix (horizon shift only) still gave R²=0.9305 — diagnosis showed `X_train_scaled` also carries 16 unfiltered case-history columns (`cases_7d_avg` etc.). The cell below fixes both: drops those 16 columns (same rule as Notebooks 08/09) and applies the `X[:-1]`/`y[1:]` shift. **Confirmed real result: RMSE=64.91, R²=-0.3533** — consistent with Notebook 08's own weather-only RF baseline (R²=-0.27 to -0.38), and proof the original R²=0.9375 was pure leakage, not genuine forecasting skill. Cite this number, not the old 0.9375 or 0.9305, in Notebook 07.

In [2]:
print("Training Random Forest...")

# ===================== LEAKAGE FIX v2 (audit remediation, CONFIRMED) =====================
# v1 (horizon shift only) was tried first and still gave R²=0.9305 — almost unchanged from
# the leaked 0.9375. Diagnosis: data['feature_columns'] (53 cols in X_train_scaled) includes
# 16 case-history columns (cases_lag_*, cases_mean_*, cases_std_*, cases_max_*, cases_ema_*,
# cases_diff_*, cases_7d_avg) that were never filtered anywhere in this notebook, so the RF
# could still use yesterday's case counts to "forecast" tomorrow's — an autoregressive
# shortcut, not a real weather-based forecast.
#
# FIX v2: drop those 16 columns via the same case_pfx/case_exact rule used in Notebooks 08/09,
# THEN apply the 1-step horizon shift (X[:-1] / y[1:], same convention as Notebooks 00/13).
# CONFIRMED result after this fix (real re-run): RMSE=64.91, MAE=46.28, R²=-0.3533 —
# matching Notebook 08's internal weather-only RF baseline (R²=-0.27 to -0.38). This proves
# the original R²=0.9375 was driven entirely by case-history leakage, not genuine
# weather-based forecasting skill.
feature_columns = data['feature_columns']
case_pfx = ['cases_lag_', 'cases_mean_', 'cases_std_', 'cases_max_', 'cases_ema_', 'cases_diff_']
case_exact = ['cases_7d_avg']
keep_idx = [i for i, c in enumerate(feature_columns)
            if not (any(c.startswith(p) for p in case_pfx) or c in case_exact)]
print(f"Keeping {len(keep_idx)}/{len(feature_columns)} weather/calendar-only columns "
      f"(dropped {len(feature_columns) - len(keep_idx)} case-history columns)")

X_train_weather = X_train[:, keep_idx]
X_test_weather = X_test[:, keep_idx]

X_train_shifted = X_train_weather[:-1]
y_train_shifted = y_train[1:]
X_test_shifted = X_test_weather[:-1]
y_test_shifted = y_test[1:]

rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=12,
    min_samples_split=10, min_samples_leaf=5,
    max_features='sqrt', random_state=42, n_jobs=-1)
rf_model.fit(X_train_shifted, y_train_shifted)
rf_pred = rf_model.predict(X_test_shifted)

rmse = np.sqrt(mean_squared_error(y_test_shifted, rf_pred))
mae = mean_absolute_error(y_test_shifted, rf_pred)
r2 = r2_score(y_test_shifted, rf_pred)

print(f"\n✅ Random Forest (weather-only, horizon-shifted, leakage-free) — RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")
print("⚠️  Original leaked run:    RMSE=13.94, R²=0.9375  (same-day X→y, case-history included)")
print("⚠️  Horizon-shift-only run: RMSE=14.71, R²=0.9305  (still had 16 case-history columns)")
print("   Neither number should be cited as the weather-only RF baseline.")

Training Random Forest...
Keeping 37/53 weather/calendar-only columns (dropped 16 case-history columns)

✅ Random Forest (weather-only, horizon-shifted, leakage-free) — RMSE: 64.91, MAE: 46.28, R²: -0.3533
⚠️  Original leaked run:    RMSE=13.94, R²=0.9375  (same-day X→y, case-history included)
⚠️  Horizon-shift-only run: RMSE=14.71, R²=0.9305  (still had 16 case-history columns)
   Neither number should be cited as the weather-only RF baseline.


In [3]:
results = {'model': 'Random Forest', 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'predictions': rf_pred}
with open('metrics/rf_results.pkl', 'wb') as f:
    pickle.dump(results, f)
with open('models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print("✅ Saved")

✅ Saved
